# Chapter 8 &mdash; Reading Off the Lengths Accepted by Any DFA

**Concept 12 of the Chapter 8 decomposition:** *Reading Off the Lengths of Strings Accepted by Any DFA*

Map every symbol to one letter with `apply_h_dfa`, determinize and minimize; the lasso shows the lengths.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Reading-Off-Lengths/Concept-Reading-Off-Lengths.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Concept 11's trick generalises. Given **any** DFA, apply the homomorphism that maps
**every** symbol to a single letter, say `1`. The resulting machine accepts $1^n$
exactly when the original accepts *some* string of length $n$.

`apply_h_dfa(D, h)` does the mapping and returns an **NFA** (several symbols collapse
onto one, so determinism is lost). Determinize and minimize, and you get the **lasso**
whose stem and cycle are the bound $b$ and period $p$ of Concept 10.

So the length set of any regular language can be read straight off a picture &mdash;
Concept 10's theorem, made constructive.

## 2. Definitions

### The homomorphism, applied

In [ ]:
def length_machine(D, letter='1'):
    N = apply_h_dfa(D, lambda c: letter)
    return min_dfa(nfa2dfa(N))

def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))

### Reference: the true length set

In [ ]:
def length_set(D, upto):
    # Same state-set walk as Chapter 8, Concept 10.
    sig = sorted(D["Sigma"])
    cur, out = {D["q0"]}, set()
    for n in range(upto + 1):
        if cur & D["F"]: out.add(n)
        cur = {step_dfa(D, q, a) for q in cur for a in sig}
    return out

<!-- nav-strip -->

---

&larr;&nbsp;[Ch8&nbsp;11.&nbsp;Solving the Stamp Problem with a Minimal DFA: $Fr = |Q| - 2$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Stamp-Problem-By-DFA/Concept-Stamp-Problem-By-DFA.ipynb) &nbsp;&middot;&nbsp; [**Chapter 8** index](https://github.com/ganeshutah/Jove/blob/master/Chapter8/README.md) &nbsp;&middot;&nbsp; [Ch9&nbsp;1.&nbsp;The GNFA: Adding `Real_I` and `Real_F` to Stand On](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-The-GNFA/Concept-The-GNFA.ipynb)&nbsp;&rarr;

---

## 3. Tests

The collapsed machine accepts $1^n$ exactly for the achievable lengths $n$.

In [ ]:
D = re_dfa("(0+1)*1(0+1)(0+1)")
L = length_machine(D)
true_set = length_set(D, 12)
from_machine = {n for n in range(13) if accepts_dfa(L, '1'*n)}
print("true length set    :", sorted(true_set))
print("from the collapsed :", sorted(from_machine))
assert true_set == from_machine

The collapsed machine is small &mdash; it is the **lasso** of the length set.

In [ ]:
print("original minimal |Q| : %d" % len(D["Q"]))
print("length machine   |Q| : %d" % len(L["Q"]))
print("alphabet of the length machine :", sorted(L["Sigma"]))
assert L["Sigma"] == {'1'}

`apply_h_dfa` returns an **NFA**, because the homomorphism is not injective.

In [ ]:
N = apply_h_dfa(D, lambda c: '1')
print("has a Q0 key (so it is an NFA)? ", 'Q0' in N)
assert 'Q0' in N
print("|Q0| =", len(N["Q0"]), "  two 0/1 edges have become two 1-edges from one state")

Reading $b$ and $p$ straight off the lasso.

In [ ]:
def lasso(D):
    seq, seen = [], {}
    q = D["q0"]
    for n in range(len(D["Q"]) + 2):
        if q in seen: return seen[q], n - seen[q]
        seen[q] = n; q = step_dfa(D, q, '1')
    return None

for r in ["(000)*", "(111+11111)*", "(0+1)*1(0+1)(0+1)", "0*1*"]:
    Lm = length_machine(re_dfa(r))
    print("%-22s length machine |Q| = %2d, (stem, cycle) = %s"
          % (r, len(Lm["Q"]), lasso(Lm)))

Cross-check against Concept 11: the stamp language gives $Fr$ again.

In [ ]:
Lm = length_machine(re_dfa("(111+11111)*"))
print("|Q| = %d, so Fr = %d  (Sylvester: %d)" % (len(Lm["Q"]), len(Lm["Q"]) - 2, 3*5-3-5))
assert len(Lm["Q"]) - 2 == 7

## 4. Animation

The length lasso for $(0+1)^*1(0+1)(0+1)$ &mdash; stem, then cycle.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(length_machine(re_dfa('(0+1)*1(0+1)(0+1)')), FuseEdges=True)

## 5. Exercises


1. Take a DFA of your own and read off its length set this way.
2. Why must the collapsed machine be nondeterministic in general?
3. How does the cycle length relate to the period $p$ of Concept 10?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter8/Concept-Reading-Off-Lengths')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')